# 26 — Cylindrical Absorption (capillary / in-situ cells)

Powder samples in capillaries or in-situ cells absorb X-rays differently
along different ray paths through the cylinder. The correction depends on
`μ·R` (linear absorption coefficient × cylinder radius) and the
scattering angle 2θ.

`CylindricalAbsorption` is a differentiable `nn.Module` that returns
the multiplicative absorption factor per ring; divide the raw integrated
intensity by this factor to recover the absorption-free profile.

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import math, numpy as np, torch, matplotlib.pyplot as plt

In [ ]:
from midas_integrate_v2.corrections import CylindricalAbsorption

# mu·R is a single dimensionless number per sample (compute it from
# composition + density + wavelength).
absorption = CylindricalAbsorption(mu_R=0.5)   # typical capillary

# Apply on a per-fit (R, two_theta) tensor
R_px = torch.linspace(100, 1800, 200, dtype=torch.float64)
two_theta = torch.atan(R_px * 200.0 / 940000.0)   # 2theta in rad
A = absorption(two_theta=two_theta)
print(f'absorption factor at first R: {float(A[0]):.4f}')
print(f'absorption factor at last R:  {float(A[-1]):.4f}  (varies with 2θ)')

## When to use it

* **Always** for capillary measurements where μR > 0.1.
* For thin films or transmission powder pellets, the correction is closer
  to a flat-plate model — different module (not yet ported).

Estimate `μR` from:
```python
mu = sum(rho_i · (mu/rho)_i for i in elements)
mu_R = mu * R_capillary_cm
```